# Project 5: Random Forest Classifier - Customer Churn Prediction

This notebook demonstrates building, tuning, and evaluating a **Random Forest Classifier** to predict customer churn. We walk through:
1. Data loading and initial exploration.
2. Training and testing set preparation using pre-split data (`churn-bigml-80.csv` and `churn-bigml-20.csv`).
3. Preprocessing (Imputation & scaling for numericals, encoding for categoricals).
4. Baseline Random Forest model training and evaluation.
5. Hyperparameter tuning using GridSearchCV with 5-fold cross-validation.
6. Model evaluation, performance comparison, and feature importance analysis.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Configure styles
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Data Acquisition

We load the customer churn datasets. The dataset is already split into training (`churn-bigml-80.csv`) and testing (`churn-bigml-20.csv`) sets. We keep these splits separate to maintain the integrity of the evaluation.

In [ ]:
# Load datasets
train_df = pd.read_csv('../data/churn-bigml-80.csv')
test_df = pd.read_csv('../data/churn-bigml-20.csv')
print(f"Training set dimensions: {train_df.shape}")
print(f"Testing set dimensions: {test_df.shape}")

## 2. Exploratory Data Analysis & Target Split

We inspect dataset properties, check for missing values, identify features/target columns, and convert the target `Churn` column to integers (`0`/`1`). We also ensure that the `Area code` categorical column is treated as a string.

In [ ]:
# Features definition
num_cols = [
    'Account length', 'Number vmail messages', 'Total day minutes', 'Total day calls',
    'Total day charge', 'Total eve minutes', 'Total eve calls', 'Total eve charge',
    'Total night minutes', 'Total night calls', 'Total night charge', 'Total intl minutes',
    'Total intl calls', 'Total intl charge', 'Customer service calls'
]
cat_cols = ['State', 'Area code', 'International plan', 'Voice mail plan']
selected_features = num_cols + cat_cols
target = 'Churn'

# Separate features and target
X_train = train_df[selected_features].copy()
y_train = train_df[target].astype(int)
X_test = test_df[selected_features].copy()
y_test = test_df[target].astype(int)

# Convert Area code to string
X_train['Area code'] = X_train['Area code'].astype(str)
X_test['Area code'] = X_test['Area code'].astype(str)

print("Missing values in training features:")
print(X_train.isnull().sum())

## 3. Preprocessing Pipeline

We create a scikit-learn `ColumnTransformer` to handle preprocessing:
- Numerical features are imputed using the **median** and scaled using **`StandardScaler`**.
- Categorical features are imputed using the **mode** and encoded using **`OneHotEncoder`**.

In [ ]:
# Numerical Transformer
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical Transformer
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

# Fit and transform
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

# Get feature names after preprocessing
num_feature_names = num_cols
cat_encoder = preprocessor.named_transformers_['cat'].named_steps['onehot']
cat_feature_names = cat_encoder.get_feature_names_out(cat_cols).tolist()
feature_names = num_feature_names + cat_feature_names
print(f"Features count after preprocessing: {len(feature_names)}")

## 4. Baseline Random Forest Model

We train a default `RandomForestClassifier` and evaluate its performance on the test set.

In [ ]:
# Train baseline model
baseline_rf = RandomForestClassifier(random_state=42)
baseline_rf.fit(X_train_preprocessed, y_train)

# Predict
y_pred_baseline = baseline_rf.predict(X_test_preprocessed)

# Metrics
baseline_metrics = {
    'Accuracy': accuracy_score(y_test, y_pred_baseline),
    'Precision': precision_score(y_test, y_pred_baseline, zero_division=0),
    'Recall': recall_score(y_test, y_pred_baseline, zero_division=0),
    'F1-score': f1_score(y_test, y_pred_baseline, zero_division=0)
}

print("Baseline Model Performance:")
for m, val in baseline_metrics.items():
    print(f" - {m}: {val:.4f}")

print("\nClassification Report (Baseline):")
print(classification_report(y_test, y_pred_baseline))

## 5. Baseline Confusion Matrix

We visualize the confusion matrix for the baseline model.

In [ ]:
cm_baseline = confusion_matrix(y_test, y_pred_baseline)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_baseline, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Non-Churn', 'Churn'],
            yticklabels=['Non-Churn', 'Churn'])
plt.title('Baseline Random Forest Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## 6. Hyperparameter Tuning using Cross-Validation

We search for the best Random Forest hyperparameters using `GridSearchCV` and 5-fold cross-validation, optimizing for F1-score.

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_preprocessed, y_train)
print("\nBest Hyperparameters Found:")
for param, val in grid_search.best_params_.items():
    print(f" - {param}: {val}")
print(f"Best CV F1-Score: {grid_search.best_score_:.4f}")

## 7. Tuned Model Evaluation

We evaluate the optimized model on the test set.

In [ ]:
tuned_rf = grid_search.best_estimator_
y_pred_tuned = tuned_rf.predict(X_test_preprocessed)

tuned_metrics = {
    'Accuracy': accuracy_score(y_test, y_pred_tuned),
    'Precision': precision_score(y_test, y_pred_tuned, zero_division=0),
    'Recall': recall_score(y_test, y_pred_tuned, zero_division=0),
    'F1-score': f1_score(y_test, y_pred_tuned, zero_division=0)
}

print("Tuned Model Performance:")
for m, val in tuned_metrics.items():
    print(f" - {m}: {val:.4f}")

print("\nClassification Report (Tuned Model):")
print(classification_report(y_test, y_pred_tuned))

## 8. Tuned Model Confusion Matrix

We visualize the confusion matrix for the tuned model.

In [ ]:
cm_tuned = confusion_matrix(y_test, y_pred_tuned)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_tuned, annot=True, fmt='d', cmap='Greens', cbar=False,
            xticklabels=['Non-Churn', 'Churn'],
            yticklabels=['Non-Churn', 'Churn'])
plt.title('Tuned Random Forest Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## 9. Performance Comparison

We compare the baseline and tuned models across all metrics.

In [ ]:
comparison_df = pd.DataFrame({
    'Metric': list(baseline_metrics.keys()),
    'Baseline': list(baseline_metrics.values()),
    'Tuned': list(tuned_metrics.values())
})

print(comparison_df.to_string(index=False))

comparison_melted = pd.melt(comparison_df, id_vars='Metric', var_name='Model', value_name='Score')
plt.figure(figsize=(10, 6))
ax = sns.barplot(data=comparison_melted, x='Metric', y='Score', hue='Model', palette='Set2')
plt.title('Performance Comparison: Baseline vs Tuned Random Forest')
plt.ylim(0, 1.05)
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f'{height:.4f}',
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='center',
                    xytext=(0, 9),
                    textcoords='offset points',
                    fontsize=10,
                    fontweight='semibold')
plt.ylabel('Score')
plt.show()

## 10. Feature Importance Analysis

We view the top feature importances from our tuned Random Forest model.

In [ ]:
importances = tuned_rf.feature_importances_
feat_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print(feat_importance_df.head(15).to_string(index=False))

plt.figure(figsize=(10, 8))
sns.barplot(data=feat_importance_df.head(15), x='Importance', y='Feature', palette='viridis', hue='Feature', legend=False)
plt.title('Top 15 Feature Importance Analysis (Tuned Random Forest)')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## 11. Final Discussion & Q&A

- **How do the baseline and tuned models compare?**
  - The tuned model optimizes parameters via cross-validation to maximize the F1-score. This controls overfitting and achieves balanced classification.
- **Which features are most predictive of customer churn?**
  - Numerical indicators like `Total day charge`, `Customer service calls`, `Total day minutes`, and `International plan_Yes` stand out as the primary predictors of customer churn.